In [1]:
# Importando bibliotecas
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split #para dividir os dados em treino e teste
from sklearn.compose import ColumnTransformer # para aplicar diferentes transformações em diferentes colunas do dataset
from sklearn.pipeline import Pipeline # para criar um pipeline de pré-processamento e modelagem
from sklearn.impute import SimpleImputer # para lidar com valores ausentes
from sklearn.preprocessing import OneHotEncoder # para codificar variáveis categóricas
from sklearn.preprocessing import StandardScaler # para padronizar os dados

#recarga da base
df = pd.read_csv(r'C:\Users\marciohenrique\Documents\Projetos\github\ml_stat\bases_dados\base_imoveis.csv')

In [2]:
"""Tratamento de valores nulos, para as colunas numéricas, 
vamos substituir os valores nulos pela mediana da coluna,
 e para as colunas categóricas, vamos substituir os valores nulos pela moda da coluna tudo via pipeline.
 exclusão de nulos da variavel alvo (preco_venda) """

colunas = [
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos",
    "localidade",
    "tipo"
]

val_numer = [
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos"
]

val_categ = [
    "localidade",
    "tipo"
]

df = df.dropna(subset=["preco_venda"]).copy() #copia da base de dados para não alterar a original


transf_val_num= Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # substitui valores nulos pela mediana
    ]
)


transf_val_cat= Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # substitui valores nulos pela moda
    ("onehot", OneHotEncoder(handle_unknown="ignore")) # codifica variáveis categóricas
    ]
)

In [4]:
"""Definindo as variaveis preditoras e a variável alvo
Configuração do percuntual de treino e teste, 80% para treino e 20% para teste"""
X = df[colunas]
y = df["preco_venda"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 80% treino e 20% teste

"""Preprocessamento dos dados, aplicando as transformações definidas anteriormente nas colunas numéricas e categóricas """
preprocessamento = ColumnTransformer(
    transformers=[
        ("num", transf_val_num, val_numer),
        ("cat", transf_val_cat, val_categ)
    ]
)

**Treino x Teste nos modelos**

Aqui vamos analisar se os modelos estão generalizando bem os dados ou se "decoraram os resultados"
sobretudo nos algoritimos de árvore

In [20]:
# Regressão Linear
from sklearn.linear_model import LinearRegression

modelo_linear = Pipeline(steps=[                    #definindo o pipeline de regressão linear
    ("preprocessamento", preprocessamento),
    ("modelo", LinearRegression())
    ]
)

modelo_linear.fit(X_train, y_train) #treinando o modelo de regressão linear
y_pred_linear = modelo_linear.predict(X_test) #fazendo previsões com o modelo de regressão linear

# Arvore de Decisão
#Usando a mesma base, aplição dos algoritmos de árvore, Decision Tree e Random Forest, para comparar os resultados com o modelo de regressão linear.


from sklearn.tree import DecisionTreeRegressor

max_depth = 10

modelo_arvore = Pipeline(
    steps=[
        ("preprocessamento", preprocessamento),
        (
            "modelo",
            DecisionTreeRegressor(
                max_depth=max_depth,
                random_state=42
            )
        )
    ]
)

modelo_arvore.fit(X_train, y_train)

y_pred_arvore = modelo_arvore.predict(X_test)

# Random Forest

from sklearn.ensemble import RandomForestRegressor
modelo_rf = Pipeline(
    steps=[
        ("processamento", preprocessamento),
        (
            "modelo",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)



In [17]:
y_pred_linear_train = modelo_linear.predict(X_train)
y_pred_arvore_train = modelo_arvore.predict(X_train)
y_pred_rf_train = modelo_rf.predict(X_train)

Função para não repetir código para verificar os indicadores

In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def calcular_metricas(y_real, y_pred):
    """Retorna R², MAE e RMSE para um conjunto de previsões."""
    return {
        "R²": r2_score(y_real, y_pred),
        "MAE": mean_absolute_error(y_real, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_pred)),
    }


def avaliar_modelo(nome, y_train, y_pred_train, y_test, y_pred_test):
    """
    Calcula R², MAE e RMSE no treino e no teste, e o gap de cada métrica
    em valor absoluto e em percentual.

    Convenção do gap: valor POSITIVO = piora do treino para o teste.
      - R²  (quanto maior, melhor): gap = treino - teste
      - MAE/RMSE (quanto menor, melhor): gap = teste - treino
    O gap percentual é sempre relativo ao valor de treino.
    """
    treino = calcular_metricas(y_train, y_pred_train)
    teste = calcular_metricas(y_test, y_pred_test)

    linha = {"Modelo": nome}
    for metrica in ["R²", "MAE", "RMSE"]:
        tr, te = treino[metrica], teste[metrica]
        gap = (tr - te) if metrica == "R²" else (te - tr)

        linha[f"{metrica} Treino"] = tr
        linha[f"{metrica} Teste"] = te
        linha[f"Gap {metrica}"] = gap
        linha[f"Gap {metrica} (%)"] = gap / tr * 100
    return linha


# Previsões de cada modelo (treino, teste)
previsoes = {
    "Regressão Linear": (y_pred_linear_train, y_pred_linear),
    "Árvore de Decisão": (y_pred_arvore_train, y_pred_arvore),
    "Random Forest": (y_pred_rf_train, y_pred_rf),
}

# DataFrame completo
df_comparacao = pd.DataFrame(
    [
        avaliar_modelo(nome, y_train, p_train, y_test, p_test)
        for nome, (p_train, p_test) in previsoes.items()
    ]
).set_index("Modelo")

# Visões separadas, mais fáceis de ler
df_metricas = df_comparacao.drop(columns=df_comparacao.filter(like="Gap").columns)
df_gaps_abs = df_comparacao[["Gap R²", "Gap MAE", "Gap RMSE"]]
df_gaps_pct = df_comparacao.filter(like="(%)")

display(df_metricas.round(4))
display(df_gaps_abs.round(4))
display(df_gaps_pct.round(2))

,R² Treino,R² Teste,MAE Treino,MAE Teste,RMSE Treino,RMSE Teste
Modelo,,,,,,
Regressão Linear,0.8379,0.8288,182280.6569,202419.5302,299579.3732,329888.5643
Árvore de Decisão,0.8384,0.7688,208702.3171,259312.1302,299086.8292,383307.7265
Random Forest,0.9856,0.8995,50698.2844,153691.2295,89261.3816,252716.6814


,Gap R²,Gap MAE,Gap RMSE
Modelo,,,
Regressão Linear,0.0091,20138.8732,30309.1911
Árvore de Decisão,0.0696,50609.8131,84220.8974
Random Forest,0.0861,102992.9452,163455.2998


,Gap R² (%),Gap MAE (%),Gap RMSE (%)
Modelo,,,
Regressão Linear,1.08,11.05,10.12
Árvore de Decisão,8.30,24.25,28.16
Random Forest,8.73,203.15,183.12


**Interpretando a Generalização**

- Regressão Linear: 
 - R² praticamente igual no treinamento e no teste(predição) Indica que o modelo generaliza bem, e que para dados que nunca viu tem o mesmo desempenho em relação aos dados já conhecidos, GAap quase zero.

 MAE e RMSE comprovam caracteristicas que uma reta não consegue prever, por isso erro e gap elevados

 Maiores Gaps, podem indicar mais decoreba do modelo ao invés de aprendizado

Decison Tree:
- R² praticamente o mesmo da regressão com dados conhecidos(treino)\
mas erra muito no teste 76%
Acaba sendo pior que a Regressão




Random Forest

R² com o maior GAP -- Apesar do indicativo de "decoreba" dos dados conhecidos, os demais indicadores superam RL e DT

GAP alto para modelos baseados em árvores é de certa forma esperado
NO Random Forest os valores de teste e treino aumentam em 3x, ou seja o modelo está decorando o treino
Nesse caso vale agora testar os hiperparametros, ou seja alterar o depth da arovre para testar
realizar valida cruzada para comprovar que o modelo não está dependendo de um split único